# Android Interview Preparation RAG System

## Purpose

This system helps prepare for Android developer interviews. It uses RAG (Retrieval-Augmented Generation) to answer questions based on a knowledge base covering Android, Kotlin, Coroutines, and Flow.

**Features:**
- 📚 Knowledge base with 100+ Android/Kotlin/Coroutines/Flow terms
- 🔍 Semantic search across the knowledge base
- 💬 Answer generation based on retrieved information
- 🎯 Preparation for typical interview questions

**Technologies:**
- Local models for embeddings and text generation
- Weaviate as vector database
- LangChain for RAG pipeline orchestration

## 1. Install Dependencies


In [1]:
import sys
import platform
import subprocess
import os

# Install dependencies
!"{sys.executable}" -m pip install -q weaviate-client==4.18.3 langchain==1.1.2 langchain-openai==1.1.0 langchain-community==0.4.1 python-dotenv==1.2.1 pandas==2.2.3 sentence-transformers==5.1.2 accelerate==1.12.0 huggingface-hub==0.36.0 torch

print("✅ Required libraries have been installed.")

# Shell command helpers for Docker
system = platform.system()
USE_WSL = system == "Windows"

def run_shell_command(command):
    """Universal function to run a shell command."""
    if USE_WSL:
        result = subprocess.run(
            ["wsl", "-e", "bash", "-l", "-c", command],
            capture_output=True, text=True, encoding="utf-8", errors="replace"
        )
    else:
        result = subprocess.run(
            command, shell=True, capture_output=True, text=True, encoding="utf-8", errors="replace"
        )
    return {
        "returncode": result.returncode,
        "stdout": result.stdout.strip(),
        "stderr": result.stderr.strip(),
        "success": result.returncode == 0
    }

print("✅ Shell command helpers are defined.")



[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
✅ Required libraries have been installed.
✅ Shell command helpers are defined.


## 2. Configuration

Setting up local models for embeddings and text generation.


In [2]:
# ==========================================
#        CONFIGURATION
# ==========================================

# Use local models
EMBEDDING_SOURCE = "local"
LLM_SOURCE = "local"

# Local models
# For embeddings: use open model (no authentication required)
LOCAL_EMBEDDING_MODEL_NAME = "google/embeddinggemma-300m"  # 768 dimensions

# For LLM: use lightweight model for CPU
LOCAL_LLM_MODEL_NAME = "google/gemma-3-1b-it"

# Weaviate configuration
WEAVIATE_CONTAINER_NAME = "android-interview-rag"
WEAVIATE_IMAGE = "semitechnologies/weaviate:1.33.7"
WEAVIATE_HTTP_PORT = 8081
WEAVIATE_GRPC_PORT = 50052

print(f"✅ Configuration loaded.")
print(f"   Embeddings: {EMBEDDING_SOURCE} ({LOCAL_EMBEDDING_MODEL_NAME})")
print(f"   LLM: {LLM_SOURCE} ({LOCAL_LLM_MODEL_NAME})")


✅ Configuration loaded.
   Embeddings: local (google/embeddinggemma-300m)
   LLM: local (google/gemma-3-1b-it)


## 3. Load Data

Knowledge base covering Android, Kotlin, Coroutines, and Flow.


In [3]:
dataset = [
    {
      "title": "Android",
      "description": "A Linux-based operating system for mobile devices developed by Google. It provides a rich application framework that allows developers to build innovative apps and games for mobile devices. Android is open-source and powers billions of devices worldwide, from smartphones to tablets, wearables, and TVs.",
      "tag": "android"
    },
    {
      "title": "Activity",
      "description": "An Android component that represents a single screen with a user interface. Each Activity is independent and can be started by other components or applications. Activities have their own lifecycle with methods like onCreate, onStart, onResume, onPause, onStop, and onDestroy that manage their state.",
      "tag": "android"
    },
    {
      "title": "Fragment",
      "description": "A modular section of an Activity that has its own lifecycle and UI. Fragments can be reused across multiple activities and allow for more flexible UI designs. They enable building multi-pane layouts and can be added, removed, or replaced dynamically during runtime.",
      "tag": "android"
    },
    {
      "title": "Service",
      "description": "A component for performing long-running operations in the background without a UI. Services continue running even when the user switches to another app. There are different types including foreground services, background services, and bound services for client-server interactions.",
      "tag": "android"
    },
    {
      "title": "BroadcastReceiver",
      "description": "A component that receives and responds to system-wide or app-specific broadcast messages. It allows apps to react to system events like battery low, network changes, or custom events. Receivers can be registered statically in the manifest or dynamically at runtime.",
      "tag": "android"
    },
    {
      "title": "ContentProvider",
      "description": "A component that manages access to a structured set of data. It provides a standardized interface for sharing data between applications securely. Content providers handle data storage using databases, files, or network resources and support CRUD operations.",
      "tag": "android"
    },
    {
      "title": "Intent",
      "description": "A messaging object used to request an action from another app component or pass data between them. Intents can be explicit (targeting a specific component) or implicit (letting the system find suitable components). They are fundamental for component communication and app navigation.",
      "tag": "android"
    },
    {
      "title": "Context",
      "description": "An interface that provides access to application-specific resources and system services. It serves as a bridge between app components and the Android system. Context is used for accessing databases, preferences, resources, and starting activities or services.",
      "tag": "android"
    },
    {
      "title": "Application",
      "description": "The base class for maintaining global application state. It's created before any other component and lives throughout the app's lifetime. Developers can extend this class to store app-wide data and initialize libraries or dependencies.",
      "tag": "android"
    },
    {
      "title": "Manifest",
      "description": "An XML file that contains essential metadata about the app including components, permissions, and features. The system reads this file to understand what components the app contains and what permissions it needs. Every Android app must have a manifest file at the root of the project.",
      "tag": "android"
    },
    {
      "title": "View",
      "description": "The base class for all UI elements in Android. Every UI component like buttons, text fields, and images extends View. Views are responsible for drawing themselves and handling user interactions like touch events.",
      "tag": "android"
    },
    {
      "title": "ViewGroup",
      "description": "A special View that can contain other Views as children. It serves as a container for organizing and positioning child views. Common ViewGroups include LinearLayout, RelativeLayout, and ConstraintLayout which define different layout strategies.",
      "tag": "android"
    },
    {
      "title": "Layout",
      "description": "Defines the visual structure and arrangement of the user interface. Layouts can be defined in XML files or programmatically in code. They determine how views are positioned and sized relative to each other and the parent container.",
      "tag": "android"
    },
    {
      "title": "RecyclerView",
      "description": "A flexible and efficient widget for displaying large lists of data. It recycles view holders to minimize memory usage and improve scrolling performance. RecyclerView requires an Adapter to bind data and a LayoutManager to position items.",
      "tag": "android"
    },
    {
      "title": "ViewModel",
      "description": "An architecture component that stores and manages UI-related data in a lifecycle-conscious way. ViewModels survive configuration changes like screen rotations, preventing data loss. They help separate UI logic from business logic and facilitate communication between fragments.",
      "tag": "android"
    },
    {
      "title": "LiveData",
      "description": "An observable data holder class that is lifecycle-aware. It automatically updates UI components when data changes and only notifies active observers. LiveData respects the lifecycle of app components, preventing memory leaks and crashes.",
      "tag": "android"
    },
    {
      "title": "Room",
      "description": "A persistence library that provides an abstraction layer over SQLite. Room makes database access more robust by providing compile-time verification of SQL queries. It consists of three main components: Database, Entity, and DAO (Data Access Object).",
      "tag": "android"
    },
    {
      "title": "WorkManager",
      "description": "An API for scheduling deferrable and guaranteed background tasks. It handles background work that needs to run reliably even if the app exits or device restarts. WorkManager chooses the appropriate way to run tasks based on device API level and app state.",
      "tag": "android"
    },
    {
      "title": "Jetpack Compose",
      "description": "A modern declarative UI toolkit for building native Android interfaces. It simplifies UI development by allowing developers to describe what the UI should look like rather than how to construct it. Compose uses Kotlin and reduces boilerplate code significantly compared to traditional XML layouts.",
      "tag": "android"
    },
    {
      "title": "Navigation Component",
      "description": "A framework for managing navigation between app destinations. It provides a consistent navigation experience and handles fragment transactions, back stack management, and deep links. The Navigation component uses a navigation graph to visualize app's navigation structure.",
      "tag": "android"
    },
    {
      "title": "Data Binding",
      "description": "A library that allows binding UI components in layouts to data sources declaratively. It reduces boilerplate code by eliminating findViewById calls and enables binding expressions in XML. Data Binding supports two-way binding for automatic UI updates when data changes.",
      "tag": "android"
    },
    {
      "title": "Permissions",
      "description": "A security system that controls app access to sensitive device features and user data. Apps must declare permissions in the manifest and request them at runtime for dangerous permissions. The permission model helps protect user privacy and device security.",
      "tag": "android"
    },
    {
      "title": "Resources",
      "description": "External files like strings, images, and layouts that are separated from code. Resources support localization, different screen sizes, and configurations without code changes. The Android system automatically selects appropriate resources based on device configuration.",
      "tag": "android"
    },
    {
      "title": "Kotlin",
      "description": "A statically-typed programming language for JVM, Android, and other platforms. Kotlin is fully interoperable with Java and offers modern language features like null safety and coroutines. It's officially supported by Google as a first-class language for Android development since 2017.",
      "tag": "kotlin"
    },
    {
      "title": "val",
      "description": "A keyword for declaring read-only (immutable) variables in Kotlin. Once assigned, the value cannot be changed, promoting functional programming patterns. Using val is preferred over var when the variable doesn't need to be reassigned.",
      "tag": "kotlin"
    },
    {
      "title": "var",
      "description": "A keyword for declaring mutable variables in Kotlin. The value can be reassigned multiple times during the variable's lifetime. While useful, overusing var can lead to code that's harder to reason about and debug.",
      "tag": "kotlin"
    },
    {
      "title": "data class",
      "description": "A class primarily designed for holding data with automatically generated utility methods. The compiler generates equals, hashCode, toString, copy, and componentN functions. Data classes reduce boilerplate and are perfect for modeling immutable data transfer objects.",
      "tag": "kotlin"
    },
    {
      "title": "sealed class",
      "description": "A class with a restricted hierarchy where all subclasses are known at compile time. Sealed classes enable exhaustive when expressions since all possible types are defined. They're ideal for representing restricted class hierarchies like state or result types.",
      "tag": "kotlin"
    },
    {
      "title": "object",
      "description": "A keyword that creates a singleton instance in Kotlin. The object is lazily initialized when first accessed and exists for the application's lifetime. Objects are useful for utility functions, constants, and implementing singleton patterns without boilerplate.",
      "tag": "kotlin"
    },
    {
      "title": "companion object",
      "description": "A special object declared inside a class that serves as a container for static members. It allows defining methods and properties that belong to the class rather than instances. Companion objects can implement interfaces and be accessed by the class name.",
      "tag": "kotlin"
    },
    {
      "title": "extension function",
      "description": "A function that adds new functionality to an existing class without inheritance or modification. Extension functions are resolved statically and don't actually modify the class. They provide a clean way to extend third-party library classes with custom methods.",
      "tag": "kotlin"
    },
    {
      "title": "lambda",
      "description": "An anonymous function that can be treated as a value and passed as an argument. Lambdas enable functional programming patterns and make code more concise. They're particularly useful with higher-order functions like map, filter, and forEach.",
      "tag": "kotlin"
    },
    {
      "title": "higher-order function",
      "description": "A function that accepts functions as parameters or returns a function as its result. They enable functional programming patterns and code reusability. Common examples include map, filter, reduce, and custom DSL builders.",
      "tag": "kotlin"
    },
    {
      "title": "null safety",
      "description": "Kotlin's type system that distinguishes between nullable and non-nullable types to prevent NullPointerException. Non-nullable types cannot hold null values, while nullable types (marked with ?) can. The compiler enforces null checks before accessing nullable values.",
      "tag": "kotlin"
    },
    {
      "title": "Elvis operator (?:)",
      "description": "An operator that provides a default value when the left operand is null. It's a concise alternative to if-else null checks. The syntax is: val result = nullableValue ?: defaultValue.",
      "tag": "kotlin"
    },
    {
      "title": "safe call (?.)",
      "description": "An operator that safely calls a method or accesses a property on a nullable object. If the object is null, the expression returns null instead of throwing an exception. Safe calls can be chained: obj?.property?.method().",
      "tag": "kotlin"
    },
    {
      "title": "lateinit",
      "description": "A modifier for delaying initialization of non-nullable properties until after construction. It's useful when dependency injection or lifecycle methods will initialize the property. Accessing a lateinit property before initialization throws an exception.",
      "tag": "kotlin"
    },
    {
      "title": "lazy",
      "description": "A delegate that defers property initialization until first access. The initialization logic runs only once and the result is cached. Lazy initialization is thread-safe by default and helps improve performance by deferring expensive computations.",
      "tag": "kotlin"
    },
    {
      "title": "scope functions (let, run, with, apply, also)",
      "description": "Functions that execute a code block within the context of an object. They differ in how they reference the context (this vs it) and what they return. Scope functions make code more concise and readable when performing operations on objects.",
      "tag": "kotlin"
    },
    {
      "title": "inline function",
      "description": "A function whose code is inserted directly at the call site during compilation. This eliminates function call overhead and enables certain optimizations. Inline functions are particularly useful with lambda parameters to avoid object allocation.",
      "tag": "kotlin"
    },
    {
      "title": "reified type parameter",
      "description": "A type parameter that is accessible at runtime in inline functions. Normally, generic type information is erased during compilation, but reified preserves it. This enables operations like instanceof checks and class references with generic types.",
      "tag": "kotlin"
    },
    {
      "title": "destructuring declaration",
      "description": "A syntax that unpacks an object into multiple variables in a single expression. Data classes automatically support destructuring through componentN functions. Example: val (name, age) = person extracts properties into separate variables.",
      "tag": "kotlin"
    },
    {
      "title": "when expression",
      "description": "A powerful alternative to switch statements for matching values against multiple conditions. When expressions are exhaustive for sealed classes and enums, ensuring all cases are handled. They can match values, ranges, types, and arbitrary boolean conditions.",
      "tag": "kotlin"
    },
    {
      "title": "range",
      "description": "A sequence of values with defined start and end points. Ranges support iteration and membership checks using the in operator. Common examples include 1..10 for inclusive ranges and 1 until 10 for exclusive end ranges.",
      "tag": "kotlin"
    },
    {
      "title": "operator overloading",
      "description": "The ability to define custom behavior for standard operators on user-defined types. Operators like +, -, *, [], and () can be overloaded using operator functions. This enables creating intuitive DSLs and more expressive APIs.",
      "tag": "kotlin"
    },
    {
      "title": "Coroutines",
      "description": "Lightweight threads for asynchronous programming in Kotlin. Coroutines can be suspended without blocking threads, making concurrent code more efficient. They simplify async code by allowing developers to write sequential-looking code that executes asynchronously.",
      "tag": "coroutines"
    },
    {
      "title": "suspend function",
      "description": "A function that can suspend execution without blocking the thread it's running on. Suspend functions can only be called from other suspend functions or coroutines. They enable writing asynchronous code in a sequential, imperative style.",
      "tag": "coroutines"
    },
    {
      "title": "CoroutineScope",
      "description": "An interface that defines the lifecycle scope of coroutines launched within it. All coroutines must run within a scope to ensure proper lifecycle management. When a scope is cancelled, all coroutines within it are cancelled too.",
      "tag": "coroutines"
    },
    {
      "title": "CoroutineContext",
      "description": "A set of elements that define coroutine behavior including dispatcher, job, and exception handler. Each coroutine has an associated context that determines where and how it executes. Contexts can be combined using the + operator.",
      "tag": "coroutines"
    },
    {
      "title": "Job",
      "description": "A cancellable handle to a coroutine's lifecycle. Jobs can be used to wait for completion, cancel execution, or manage parent-child relationships. Every coroutine has an associated Job that tracks its state.",
      "tag": "coroutines"
    },
    {
      "title": "Dispatchers",
      "description": "Determine which thread or thread pool a coroutine runs on. The choice of dispatcher affects performance and behavior of coroutines. Kotlin provides several built-in dispatchers optimized for different use cases.",
      "tag": "coroutines"
    },
    {
      "title": "Dispatchers.Main",
      "description": "A dispatcher that executes coroutines on the main UI thread. It's essential for updating UI elements safely in Android and other UI frameworks. Operations on Main should be quick to avoid blocking user interactions.",
      "tag": "coroutines"
    },
    {
      "title": "Dispatchers.IO",
      "description": "A dispatcher optimized for I/O operations like network requests, file access, and database queries. It uses a shared pool of threads designed to handle blocking operations efficiently. The thread pool grows as needed to accommodate blocked operations.",
      "tag": "coroutines"
    },
    {
      "title": "Dispatchers.Default",
      "description": "A dispatcher for CPU-intensive operations like parsing, sorting, or complex calculations. It uses a shared pool sized to the number of CPU cores. Default is ideal for work that doesn't block on I/O.",
      "tag": "coroutines"
    },
    {
      "title": "launch",
      "description": "A coroutine builder that starts a new coroutine without returning a result. It returns a Job that can be used to control the coroutine's lifecycle. Launch is used for fire-and-forget operations that don't need to return values.",
      "tag": "coroutines"
    },
    {
      "title": "async",
      "description": "A coroutine builder that starts a coroutine and returns a Deferred result. It's used when you need to return a value from asynchronous work. Multiple async operations can run concurrently and their results awaited together.",
      "tag": "coroutines"
    },
    {
      "title": "Deferred",
      "description": "A non-blocking cancellable future that represents a promise to deliver a result. It's returned by async and provides an await function to retrieve the result. Deferred extends Job, so it can be cancelled like any coroutine.",
      "tag": "coroutines"
    },
    {
      "title": "await",
      "description": "A suspend function that waits for a Deferred to complete and returns its result. If the Deferred was cancelled or failed, await throws the corresponding exception. Multiple Deferred values can be awaited concurrently using awaitAll.",
      "tag": "coroutines"
    },
    {
      "title": "withContext",
      "description": "A suspend function that switches the coroutine's context for a block of code. It's commonly used to switch between Dispatchers for different types of work. withContext suspends until the block completes and returns its result.",
      "tag": "coroutines"
    },
    {
      "title": "coroutineScope",
      "description": "Creates a new coroutine scope that completes only after all launched children complete. If any child fails, the scope cancels all other children and rethrows the exception. It's useful for concurrent decomposition of work.",
      "tag": "coroutines"
    },
    {
      "title": "supervisorScope",
      "description": "A scope where failure of one child coroutine doesn't cancel other siblings or the parent. Each child's failure is isolated and must be handled individually. It's useful when you want independent operations that shouldn't affect each other.",
      "tag": "coroutines"
    },
    {
      "title": "delay",
      "description": "A suspend function that pauses coroutine execution for a specified time without blocking the thread. It's a non-blocking alternative to Thread.sleep that's cancellable. Delay is useful for implementing timeouts, polling, and animations.",
      "tag": "coroutines"
    },
    {
      "title": "yield",
      "description": "A suspend function that voluntarily gives up execution to allow other coroutines to run. It's useful in CPU-intensive operations to avoid starving other coroutines. Yield also checks for cancellation and throws CancellationException if cancelled.",
      "tag": "coroutines"
    },
    {
      "title": "runBlocking",
      "description": "Blocks the current thread until all coroutines within it complete. It's primarily used in main functions and tests to bridge blocking and suspending worlds. runBlocking should not be used in production coroutine code.",
      "tag": "coroutines"
    },
    {
      "title": "GlobalScope",
      "description": "A global coroutine scope whose lifetime spans the entire application. Coroutines launched in GlobalScope aren't bound to any lifecycle and must be cancelled manually. Using GlobalScope is generally discouraged in favor of structured concurrency.",
      "tag": "coroutines"
    },
    {
      "title": "structured concurrency",
      "description": "A principle that organizes coroutines into a hierarchy with clear lifecycle rules. Parent coroutines wait for all children to complete and cancellation propagates through the hierarchy. This prevents coroutine leaks and makes concurrent code easier to reason about.",
      "tag": "coroutines"
    },
    {
      "title": "exception handling",
      "description": "Coroutines handle exceptions through try-catch blocks and CoroutineExceptionHandler. Uncaught exceptions cancel the parent scope by default. Proper exception handling is crucial for building robust asynchronous applications.",
      "tag": "coroutines"
    },
    {
      "title": "CoroutineExceptionHandler",
      "description": "A context element that handles uncaught exceptions in coroutines. It's invoked for exceptions that aren't caught by try-catch blocks. Exception handlers are typically used for logging or reporting crashes.",
      "tag": "coroutines"
    },
    {
      "title": "SupervisorJob",
      "description": "A Job that doesn't cancel its parent or siblings when it fails. It's used as the Job in supervisorScope and enables independent error handling for each child. SupervisorJob is useful when you want failures to be isolated.",
      "tag": "coroutines"
    },
    {
      "title": "cancellation",
      "description": "The mechanism for interrupting and stopping coroutine execution. Cancellation is cooperative, meaning coroutines must check for cancellation and respond appropriately. Cancelled coroutines throw CancellationException which shouldn't be caught and suppressed.",
      "tag": "coroutines"
    },
    {
      "title": "isActive",
      "description": "A property that indicates whether a coroutine is still active and not cancelled. Long-running loops should check isActive periodically to respect cancellation. It's available in any CoroutineScope.",
      "tag": "coroutines"
    },
    {
      "title": "ensureActive",
      "description": "A function that checks if the coroutine is active and throws CancellationException if cancelled. It's more explicit than checking isActive manually. ensureActive is useful for making non-suspending code cancellable.",
      "tag": "coroutines"
    },
    {
      "title": "Flow",
      "description": "A cold asynchronous stream that emits values sequentially over time. Flows are built on top of coroutines and support backpressure naturally. They're similar to sequences but designed for asynchronous operations.",
      "tag": "flow"
    },
    {
      "title": "cold stream",
      "description": "A stream that only starts emitting values when someone collects from it. Each collector gets its own independent stream of values. Cold streams are lazy and don't do any work until observed.",
      "tag": "flow"
    },
    {
      "title": "hot stream",
      "description": "A stream that emits values regardless of whether anyone is collecting. Multiple collectors can observe the same stream of values. Hot streams like StateFlow and SharedFlow are useful for sharing state.",
      "tag": "flow"
    },
    {
      "title": "flow builder",
      "description": "A function that creates a Flow from a suspend lambda with emit calls. The builder doesn't execute any code until the flow is collected. It's the fundamental way to create custom flows.",
      "tag": "flow"
    },
    {
      "title": "emit",
      "description": "A suspend function that sends a value downstream in a Flow. Emit suspends until the value is processed by the collector. It can only be called from within a flow builder or other flow contexts.",
      "tag": "flow"
    },
    {
      "title": "collect",
      "description": "A terminal operator that subscribes to a Flow and receives its values. Collect suspends until the flow completes or is cancelled. It's the primary way to consume values from a flow.",
      "tag": "flow"
    },
    {
      "title": "flowOf",
      "description": "A builder function that creates a Flow from a fixed set of values. It emits all values sequentially when collected. flowOf is useful for testing and creating simple flows.",
      "tag": "flow"
    },
    {
      "title": "asFlow",
      "description": "An extension function that converts collections, sequences, and ranges to Flow. It provides an easy way to work with existing data structures as flows. The conversion is cold and creates a new flow on each call.",
      "tag": "flow"
    },
    {
      "title": "map",
      "description": "An operator that transforms each value emitted by the Flow. The transformation function is applied to every value. map is one of the most commonly used flow operators.",
      "tag": "flow"
    },
    {
      "title": "filter",
      "description": "An operator that only emits values that match a given predicate. Values that don't satisfy the condition are dropped. filter is useful for selecting specific values from a stream.",
      "tag": "flow"
    },
    {
      "title": "transform",
      "description": "A universal operator for arbitrary flow transformations. Unlike map, transform can emit zero, one, or multiple values for each input. It provides maximum flexibility for custom flow operations.",
      "tag": "flow"
    },
    {
      "title": "take",
      "description": "An operator that limits the number of emitted values. After emitting the specified count, the flow completes. take is useful for creating bounded flows from potentially infinite sources.",
      "tag": "flow"
    },
    {
      "title": "drop",
      "description": "An operator that skips the first N values from the Flow. Subsequent values are emitted normally. drop is useful for ignoring initial values in a stream.",
      "tag": "flow"
    },
    {
      "title": "flatMapConcat",
      "description": "An operator that transforms each value into a Flow and concatenates them sequentially. Each inner flow must complete before the next one starts. It preserves the order of operations.",
      "tag": "flow"
    },
    {
      "title": "flatMapMerge",
      "description": "An operator that transforms values into Flows and merges them concurrently. Multiple inner flows can execute simultaneously up to the concurrency limit. Values may arrive out of order.",
      "tag": "flow"
    },
    {
      "title": "zip",
      "description": "An operator that combines two Flows by pairing their values. Each pair consists of corresponding values from both flows. The resulting flow completes when either source completes.",
      "tag": "flow"
    },
    {
      "title": "combine",
      "description": "An operator that combines the latest values from multiple Flows. Whenever any flow emits, the combiner function is called with the latest values from all flows. It's useful for UI state that depends on multiple sources.",
      "tag": "flow"
    },
    {
      "title": "debounce",
      "description": "An operator that filters out values emitted too quickly. Only the latest value within a time window is emitted. debounce is ideal for search queries or other user input that shouldn't trigger on every keystroke.",
      "tag": "flow"
    },
    {
      "title": "distinctUntilChanged",
      "description": "An operator that filters out consecutive duplicate values. It only emits when the value differs from the previous one. This operator reduces unnecessary updates when the same value is emitted repeatedly.",
      "tag": "flow"
    },
    {
      "title": "onEach",
      "description": "An operator that performs an action on each emitted value without modifying the stream. It's useful for side effects like logging or analytics. onEach doesn't affect the values flowing downstream.",
      "tag": "flow"
    },
    {
      "title": "catch",
      "description": "An operator that handles exceptions from upstream Flow operations. It can emit fallback values or rethrow different exceptions. catch doesn't handle exceptions from downstream operations.",
      "tag": "flow"
    },
    {
      "title": "onCompletion",
      "description": "An operator that executes an action when the Flow completes, whether normally or exceptionally. It receives the completion cause (null for normal completion, exception otherwise). onCompletion is useful for cleanup operations.",
      "tag": "flow"
    },
    {
      "title": "flowOn",
      "description": "An operator that changes the CoroutineContext for upstream flow operations. It creates a buffer between producers and consumers on different dispatchers. flowOn only affects operations above it in the flow chain.",
      "tag": "flow"
    },
    {
      "title": "buffer",
      "description": "An operator that creates a buffer between producer and consumer coroutines. It allows the producer to continue emitting while the consumer processes previous values. buffer improves throughput when production and consumption have different speeds.",
      "tag": "flow"
    },
    {
      "title": "conflate",
      "description": "An operator that skips intermediate values when the consumer is slower than the producer. Only the latest value is kept in the buffer. conflate is useful when you only care about the most recent state.",
      "tag": "flow"
    },
    {
      "title": "StateFlow",
      "description": "A hot Flow that always holds the current state value. It's similar to LiveData but built on Flow and always has a value. StateFlow is ideal for representing UI state that can be observed by multiple collectors.",
      "tag": "flow"
    },
    {
      "title": "SharedFlow",
      "description": "A hot Flow for multicasting values to multiple subscribers. Unlike StateFlow, it doesn't hold a current value and can be configured with replay cache. SharedFlow is useful for events or commands.",
      "tag": "flow"
    },
    {
      "title": "MutableStateFlow",
      "description": "A mutable version of StateFlow that allows updating the current value. It provides thread-safe updates through the value property. MutableStateFlow is commonly used in ViewModels to expose mutable state internally.",
      "tag": "flow"
    },
    {
      "title": "MutableSharedFlow",
      "description": "A mutable version of SharedFlow that allows emitting values programmatically. It supports suspending emit and non-suspending tryEmit operations. MutableSharedFlow is useful for implementing event buses.",
      "tag": "flow"
    },
    {
      "title": "stateIn",
      "description": "An operator that converts a cold Flow into a hot StateFlow. It requires a scope, started strategy, and initial value. stateIn is useful for sharing flow results among multiple collectors.",
      "tag": "flow"
    },
    {
      "title": "shareIn",
      "description": "An operator that converts a cold Flow into a hot SharedFlow. It allows configuring replay cache size and start behavior. shareIn is useful for sharing expensive flow operations.",
      "tag": "flow"
    },
    {
      "title": "channelFlow",
      "description": "A builder that creates a Flow with concurrent emission capabilities. Unlike regular flow builders, it allows emitting from multiple coroutines. channelFlow bridges channels and flows for complex scenarios.",
      "tag": "flow"
    }
]

# Convert dataset to RAG format
documents_data = []
for item in dataset:
    # Combine title and description into content for better search
    content = f"{item['title']}: {item['description']}"
    documents_data.append({
        "title": item["title"],
        "content": content,
        "tag": item.get("tag", "android")
    })

print(f"✅ Loaded {len(documents_data)} documents from Android knowledge base.")
print(f"   Topics: {set([doc['tag'] for doc in documents_data])}")


✅ Loaded 104 documents from Android knowledge base.
   Topics: {'kotlin', 'coroutines', 'android', 'flow'}


## 4. Start Weaviate (Vector Database)


In [4]:
# Stop old container if exists
print(f"--- Stopping and removing any existing container named '{WEAVIATE_CONTAINER_NAME}' ---")
stop_command = f"docker stop {WEAVIATE_CONTAINER_NAME} 2>/dev/null; docker rm {WEAVIATE_CONTAINER_NAME} 2>/dev/null"
run_shell_command(stop_command)
print("Cleanup complete.")

# Start new Weaviate container
print(f"\n--- Starting Weaviate container '{WEAVIATE_CONTAINER_NAME}' ---")
run_command = (
    f"docker run -d "
    f"--name {WEAVIATE_CONTAINER_NAME} "
    f"-p {WEAVIATE_HTTP_PORT}:8080 "
    f"-p {WEAVIATE_GRPC_PORT}:50051 "
    f"-e AUTHENTICATION_ANONYMOUS_ACCESS_ENABLED=true "
    f"-e PERSISTENCE_DATA_PATH=/var/lib/weaviate "
    f"-e DEFAULT_VECTORIZER_MODULE=none "
    f"-e ENABLE_MODULES='' "
    f"-e CLUSTER_HOSTNAME=node1 "
    f"{WEAVIATE_IMAGE}"
)

result = run_shell_command(run_command)

if result["success"]:
    print("✅ Weaviate container started successfully.")
    print("Waiting a few seconds for the service to initialize...")
    import time
    time.sleep(10)
else:
    print("❌ Failed to start Weaviate container.")
    print(f"Stderr: {result['stderr']}")


--- Stopping and removing any existing container named 'android-interview-rag' ---
Cleanup complete.

--- Starting Weaviate container 'android-interview-rag' ---
✅ Weaviate container started successfully.
Waiting a few seconds for the service to initialize...


## 5. Setup Models and Generate Embeddings


In [5]:
import weaviate
import weaviate.classes as wvc
from weaviate.util import generate_uuid5

from rag_models import LocalHuggingFaceEmbeddings

# --- Initialize models ---
print("--- 1. Setting up embeddings model ---")
try:
    embeddings_model = LocalHuggingFaceEmbeddings(LOCAL_EMBEDDING_MODEL_NAME)
    print("✅ Initialized.")
except Exception as e:
    print(f"❌ Failed to initialize: {e}")
    raise

# --- Generate embeddings ---
print("\n--- 2. Generating embeddings for all documents ---")
contents_to_embed = [doc['content'] for doc in documents_data]
vector_embeddings = embeddings_model.embed_documents(contents_to_embed)
print(f"✅ Generated {len(vector_embeddings)} embeddings. Vector dimension: {len(vector_embeddings[0])}")

# Add embeddings to documents
for i, doc in enumerate(documents_data):
    doc['content_vector'] = vector_embeddings[i]

# --- Connect to Weaviate ---
print("\n--- 3. Connecting to Weaviate ---")
weaviate_client = weaviate.connect_to_local(
    host="localhost",
    port=WEAVIATE_HTTP_PORT,
    grpc_port=WEAVIATE_GRPC_PORT
)

if weaviate_client.is_ready():
    print("✅ Successfully connected to Weaviate.")
else:
    print("❌ Failed to connect to Weaviate.")
    weaviate_client.close()
    raise ConnectionError("Could not connect to Weaviate instance.")

# --- Create collection ---
COLLECTION_NAME = "AndroidInterview"
print(f"\n--- 4. Creating Weaviate collection: '{COLLECTION_NAME}' ---")

if weaviate_client.collections.exists(COLLECTION_NAME):
    weaviate_client.collections.delete(COLLECTION_NAME)
    print(f"Deleted existing collection '{COLLECTION_NAME}'.")

rag_collection = weaviate_client.collections.create(
    name=COLLECTION_NAME,
    properties=[
        wvc.config.Property(name="title", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="content", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="tag", data_type=wvc.config.DataType.TEXT),
    ],
    vector_config=wvc.config.Configure.Vectors.self_provided(
        vector_index_config=wvc.config.Configure.VectorIndex.hnsw(
            distance_metric=wvc.config.VectorDistances.COSINE
        )
    )
)
print(f"✅ Collection '{COLLECTION_NAME}' created successfully.")

# --- Ingest data into Weaviate ---
print(f"\n--- 5. Ingesting {len(documents_data)} documents into Weaviate ---")
with rag_collection.batch.dynamic() as batch:
    for doc in documents_data:
        properties = {
            "title": doc["title"],
            "content": doc["content"],
            "tag": doc["tag"]
        }
        batch.add_object(
            properties=properties,
            vector=doc["content_vector"],
            uuid=generate_uuid5(doc["title"])
        )

print(f"✅ Data ingestion complete. Total objects in collection: {len(rag_collection)}")
print("✅ RAG system is ready!")
weaviate_client.close()

/Users/nurzhaussyn/Documents/EpamGenAI/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


--- 1. Setting up embeddings model ---
📥 Loading local embedding model: google/embeddinggemma-300m...
✅ Local embedding model loaded successfully.
✅ Initialized.

--- 2. Generating embeddings for all documents ---
✅ Generated 104 embeddings. Vector dimension: 768

--- 3. Connecting to Weaviate ---
✅ Successfully connected to Weaviate.

--- 4. Creating Weaviate collection: 'AndroidInterview' ---
✅ Collection 'AndroidInterview' created successfully.

--- 5. Ingesting 104 documents into Weaviate ---
✅ Data ingestion complete. Total objects in collection: 104
✅ RAG system is ready!


## 8. Cleanup


In [6]:
# # Stop and remove container
# print(f"--- Stopping and removing container '{WEAVIATE_CONTAINER_NAME}' ---")
# cleanup_command = f"docker stop {WEAVIATE_CONTAINER_NAME} && docker rm {WEAVIATE_CONTAINER_NAME}"
# result = run_shell_command(cleanup_command)

# if result["success"]:
#     print(f"✅ Container stopped and removed successfully.")
# else:
#     print(f"⚠️ Container might have already been stopped.")
#     print(f"Stderr: {result['stderr']}")
